# Load data of location and net rated power of wind turbines and solar stations

In this notebook weights are build to aggregate weather data for the whole bidding zone DE-LU during the downlaod process of weather forecasts (used in noaa_gfs.py).

The data is from https://www.marktstammdatenregister.de/MaStR/Datendownload.

In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
import geopandas as gpd
from pathlib import Path

## Load wind data

In [33]:
def local(tag): return tag.split("}", 1)[-1]

cols_wind = [
    "EinheitMastrNummer","Energietraeger","WindAnLandOderAufSee","Nettonennleistung",
    "Breitengrad","Laengengrad","Gemeinde","Gemeindeschluessel","Inbetriebnahmedatum",
    "Nabenhoehe","NameWindpark","EinheitBetriebsstatus"
]


rows = []
for _, e in ET.iterparse("Gesamtdatenexport_20260101_25.2/EinheitenWind.xml", events=("end",)):
    if local(e.tag) == "EinheitWind":
        row = {}
        for child in e:
            n = local(child.tag)
            if n in cols_wind:
                row[n] = child.text
        rows.append(row)
        e.clear()

df_wind = pd.DataFrame(rows)[[c for c in cols_wind if c in rows[0]]]

##### Filter for only turbines in operation

In [34]:
# only turbines in operation
df_wind = df_wind[df_wind["EinheitBetriebsstatus"] == "35"]

## Load solar data

In [35]:
def local(tag):
    return tag.split("}", 1)[-1]

rows = []

cols_solar = [
    "EinheitMastrNummer",
    "Energietraeger",
    "Nettonennleistung",
    "Gemeinde",
    "Gemeindeschluessel",
    "Inbetriebnahmedatum",
    "EinheitBetriebsstatus"
]

cols_set = set(cols_solar)

solar_files = list(Path("Gesamtdatenexport_20260101_25.2").glob("EinheitenSolar*.xml"))
print("Solar files:", len(solar_files))

for file in solar_files:
    print("reading", file)

    for _, e in ET.iterparse(file, events=("end",)):
        if local(e.tag) == "EinheitSolar":

            row = {}
            for child in e:
                name = local(child.tag)
                if name in cols_set:
                    row[name] = child.text

            rows.append(row)
            e.clear()

df_solar = pd.DataFrame(rows)
df_solar = df_solar[[c for c in cols_solar if c in df_solar.columns]]

Solar files: 59
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_59.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_58.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_48.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_49.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_39.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_11.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_7.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_6.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_10.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_38.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_12.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_4.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_5.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_13.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_17.xml
reading Gesamtdatenexport_20260101_25.2/EinheitenSolar_1.x

##### Filter for only solar units in operation

In [38]:
# only solar units in operation
df_solar = df_solar[df_solar["EinheitBetriebsstatus"] == "35"]

## Get lon/lat of units without this information using city information

In [40]:
lonlat_missing_wind = int((df_wind["Breitengrad"].isna() | df_wind["Laengengrad"].isna()).sum())
print(f"Missing longitude/latitude information for wind power units: {lonlat_missing_wind} of {len(df_wind)} total units.")

Missing longitude/latitude information for wind power units: 1006 of 31969 total units.


The solar data has no lon/lat information. Also some entries of the wind data have missing lon/lat information. The given city keys are used to add the coordinates to the power units.

### Download of the city keys with lon/lat information

In [ ]:
# get polygons of citys 
wfs = "https://sgx.geodatenzentrum.de/wfs_vg250"
url = f"{wfs}?service=WFS&version=2.0.0&request=GetFeature&TYPENAMES=vg250_gem"
gdf = gpd.read_file(url)

# calculate centroids for each polygon and put into dataframe
gdf["centroid"] = gdf.geometry.centroid.to_crs(4326)

map_df = pd.DataFrame({
    "Gemeindeschluessel": gdf["ags"].astype(str).str.zfill(8),
    "Breitengrad": gdf["centroid"].y,
    "Laengengrad": gdf["centroid"].x,
})

# some keys are duplicated, the first ones are kept to have a unique mapping
map_df = map_df.drop_duplicates("Gemeindeschluessel", keep="first")

### Join the power unit data with the lon/lat information via city key

In [41]:
# join solar/wind data with coordinates
df_solar["Gemeindeschluessel"] = df_solar["Gemeindeschluessel"].astype(str).str.zfill(8)
df_solar = df_solar.merge(map_df, on="Gemeindeschluessel", how="left")

df_wind["Gemeindeschluessel"] = df_wind["Gemeindeschluessel"].astype(str).str.zfill(8)
df_wind = df_wind.merge(map_df, on="Gemeindeschluessel", how="left")

For wind data only use new lon/lat information, if the information is missing:

In [42]:
# get unique lon/lat pairs for wind
df_wind["Breitengrad"] = df_wind["Breitengrad_x"].fillna(df_wind["Breitengrad_y"])
df_wind["Laengengrad"] = df_wind["Laengengrad_x"].fillna(df_wind["Laengengrad_y"])

Some entries still have no lon/lat information:

In [47]:
print(f"SOLAR - Missing lon/lat values: {int((df_solar['Breitengrad'].isna() | df_solar['Laengengrad'].isna()).sum())} of in total {len(df_solar)} units.")
print(f"WIND - Missing lon/lat values: {int((df_wind['Breitengrad'].isna() | df_wind['Laengengrad'].isna()).sum())} of in total {len(df_wind)} units.")
df_solar = df_solar.dropna(subset=["Breitengrad", "Laengengrad"])
df_wind = df_wind.dropna(subset=["Breitengrad", "Laengengrad"])

SOLAR - Missing lon/lat values: 10374 of in total 5698598 units.
WIND - Missing lon/lat values: 3 of in total 31969 units.


These entries are dropped as these are less than 0.2% of the total entries.

## Save data frames

In [48]:
# convert to numeric before saving
for df in [df_wind, df_solar]:
    for c in ["Breitengrad", "Laengengrad", "Nettonennleistung"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

In [49]:
df_wind.to_parquet("power_wind.parquet", compression="zstd", index=False)
df_solar.to_parquet("power_solar.parquet", compression="zstd", index=False)